# BCO7006 Session 5 — Object-Oriented Programming Foundations

**Week 3 · Session 5 · 3 hours**

Last week, you noticed something awkward. Every function you wrote for the order-processor mini-project ended up taking the same two dictionaries as arguments — `customer` and `cart`. Today you'll learn how Python solves this: by bundling data and the functions that work on it into a single thing called an **object**.

By the end of this session, you'll be able to:

- Define a class with attributes and methods using `class`, `__init__`, and `self`.
- Create instances of a class and call methods on them.
- Refactor a function-and-dictionary design into an equivalent class-based design.
- Use `isinstance()` to check the type of an object.
- Combine classes through **composition** to model "has-a" relationships.

## How this notebook works

Cells in this notebook come in five difficulty levels. You'll meet them in roughly this order in every section:

| Level | What it looks like | Where |
|---|---|---|
| **L1 Mirror** | Read the instructor's code — you don't retype it | In-class |
| **L2 Trace** | Predict the output **in a markdown cell**, then run | In-class |
| **L3 Modify** | Change working code or fix a planted bug | In-class |
| **L4 Build** | Build from a spec — **Claude/Google permitted** | At-home |
| **L5 Extend** | Open-ended design task — **marked pass/fail on engagement** | At-home |

**On using AI tools.** For L4 and L5, you may use Claude, ChatGPT, GitHub Copilot, or anything else you find useful. We expect you to. What we check is whether you understood what you got back. Two things matter:

1. **At the end of every L4/L5 task, the reflection cell asks you to paste your prompt and say one thing the AI got right and one thing you changed, rejected, or looked up.** Honest answers are fine — "I rejected its first attempt because it used a feature we haven't covered" is a great answer.
2. **Some cells in this notebook are explicitly AI-resistant** — they ask you to compare to *your* S4 code, or to predict before you run, or to explain what a specific `self` refers to in a specific line. These can't be pasted from a chatbot because the chatbot doesn't know your code or what's happening on your screen.

**Setup.** Open this in Google Colab. Save a copy to your drive (File → Save a copy in Drive) so your work persists.

---

# Part 1: Defining classes — refactoring the cart

Quick recap: in Session 4 you built an order processor using dictionaries and functions. Here's the shape of what you wrote (read this cell — don't change anything in it):

In [ ]:
# --- S4 reminder (read-only) ---

catalogue = {
    "P001": {"name": "Notebook", "price": 12.50},
    "P002": {"name": "Pen pack", "price": 4.20},
    "P003": {"name": "Stapler", "price": 8.90},
}

customer = {
    "name": "Alice Chen",
    "email": "alice@example.com",
    "tier": "gold",
}

cart = {
    "items": [
        {"product_id": "P001", "qty": 2},
        {"product_id": "P002", "qty": 1},
    ]
}

def compute_subtotal(cart, catalogue):
    total = 0
    for item in cart["items"]:
        price = catalogue[item["product_id"]]["price"]
        total += price * item["qty"]
    return total

print("S4 subtotal:", compute_subtotal(cart, catalogue))

Now look at that function signature: `compute_subtotal(cart, catalogue)`. The function takes the cart as an argument — even though the function is *all about* the cart. Most of your other functions had the same shape: they were *about* a thing, but they had to be *handed* the thing every time.

Today's question: **what if the thing carried its own functions around with it?**

## The S5 way — a `Cart` class (instructor demo — read)

A **class** is a template for creating objects. An **object** is one specific thing made from that template. Here is the same cart logic from above, rewritten as a class:

In [ ]:
class Cart:
    def __init__(self, items):
        self.items = items

    def compute_subtotal(self, catalogue):
        total = 0
        for item in self.items:
            price = catalogue[item["product_id"]]["price"]
            total += price * item["qty"]
        return total

Three things to notice:

1. **`class Cart:`** declares the template. Indented underneath are the things every Cart will have.
2. **`def __init__(self, items):`** is the *constructor* — Python runs this code every time you create a new Cart. The `items` parameter is the data you pass in. `self.items = items` stores it on the new Cart object.
3. **`def compute_subtotal(self, catalogue):`** is a *method* — a function that lives inside the class. Methods always have `self` as their first parameter. `self` is "the specific Cart object this is being called on."

Watch how we use it:

In [ ]:
# Create a Cart object using the same items as the S4 cart
alice_cart = Cart(items=[
    {"product_id": "P001", "qty": 2},
    {"product_id": "P002", "qty": 1},
])

# Access the data
print("Items in cart:", alice_cart.items)

# Call the method
print("Subtotal:", alice_cart.compute_subtotal(catalogue))

## Side-by-side

| S4 (procedural) | S5 (object-oriented) |
|---|---|
| `cart = {"items": [...]}` | `alice_cart = Cart(items=[...])` |
| `cart["items"]` | `alice_cart.items` |
| `compute_subtotal(cart, catalogue)` | `alice_cart.compute_subtotal(catalogue)` |

The logic inside `compute_subtotal` is **identical** — you can compare line by line. What changed is the *structure*: the function now lives inside the class, and the cart's data lives inside the cart object.

## Your turn (L3): add a `count_items` method

The Cart class needs a method `count_items(self)` that returns the **total quantity** of items in the cart — not the number of distinct products, but the total quantity across all items.

For Alice's cart above (2 notebooks + 1 pen pack), `count_items()` should return `3`.

Modify the Cart class below. Don't change `compute_subtotal` — just add a new method.

In [ ]:
class Cart:
    def __init__(self, items):
        self.items = items

    def compute_subtotal(self, catalogue):
        total = 0
        for item in self.items:
            price = catalogue[item["product_id"]]["price"]
            total += price * item["qty"]
        return total

    # TODO: add count_items(self) here


# Test your code
alice_cart = Cart(items=[
    {"product_id": "P001", "qty": 2},
    {"product_id": "P002", "qty": 1},
])
print("Total items:", alice_cart.count_items())  # should print 3

## Comprehension check

Look at these two lines from the code above:

- Line A: `self.items = items` (inside `__init__`)
- Line B: `alice_cart.compute_subtotal(catalogue)` (when we called the method)

In Line A, what does `self` refer to?
In Line B, what is the `self` that the method receives?
Are they the same thing?

*Answer in the markdown cell below by double-clicking to edit it.*

**Your answer:**

*(replace this text)*

---

# Part 2: Your turn — the Customer class

In S4 you had a customer dict and an `apply_discount(customer, total)` function. Now you'll rebuild both as a single class.

## Build it (L3a)

Write a `Customer` class with:

- A constructor that accepts `name`, `email`, and `tier`.
- An `apply_discount(self, total)` method that returns the discounted total:
  - `"gold"` tier: 15% off (return `total * 0.85`)
  - `"silver"` tier: 10% off
  - any other tier: no discount

In [ ]:
class Customer:
    def __init__(self, name, email, tier):
        # TODO: store name, email, tier as attributes
        pass

    def apply_discount(self, total):
        # TODO: return the discounted total based on self.tier
        pass


# Test your code
alice = Customer("Alice Chen", "alice@example.com", "gold")
bob = Customer("Bob Singh", "bob@example.com", "silver")
carol = Customer("Carol Mehta", "carol@example.com", "bronze")

print("Alice on $100:", alice.apply_discount(100))   # 85.0
print("Bob on $100:  ", bob.apply_discount(100))     # 90.0
print("Carol on $100:", carol.apply_discount(100))   # 100

## Find the bug (L3b)

Below is another version of the Customer class. The test at the bottom runs without raising any error — but the results are **wrong**. Run it, look at the output, then read the code and find the bug.

In [ ]:
class CustomerBuggy:
    def __init__(self, name, email, tier):
        self.name = name
        self.email = email
        self.tier = tier

    def apply_discount(self, total):
        if self.tier == "gold":
            return total * 0.15
        elif self.tier == "silver":
            return total * 0.10
        else:
            return 0


alice_b = CustomerBuggy("Alice", "a@x.com", "gold")
print("Buggy Alice on $100:", alice_b.apply_discount(100))   # what does this print?
print("What it SHOULD print: 85.0 (a 15% discount means you pay 85% of the original)")

**Your task:**

1. In a sentence, describe what the bug is. (What is `apply_discount` returning instead of what it should return?)
2. Write the corrected version in the cell below.

**Description of the bug:**

*(replace this text)*

In [ ]:
# Corrected version
class CustomerFixed:
    def __init__(self, name, email, tier):
        self.name = name
        self.email = email
        self.tier = tier

    def apply_discount(self, total):
        # TODO: write the corrected logic
        pass


# Test
print(CustomerFixed("Alice", "a@x.com", "gold").apply_discount(100))   # should print 85.0
print(CustomerFixed("Bob",   "b@x.com", "silver").apply_discount(100)) # should print 90.0

## Comprehension check

Two uses of `self` from the code above:

- In `self.tier = tier`, what is `self` referring to?
- In `customer.apply_discount(100)`, where the method receives `self` as its first parameter — what is that `self`?

Are they the same kind of thing? Why does Python use the same word for both?

**Your answer:**

*(replace this text)*

## At-home — Claude allowed (L4): `LoyaltyCustomer`

**You will need to complete this before the next session.** Use Claude, Google, or whatever you find useful — but read the reflection prompt at the end before you start.

Build a `LoyaltyCustomer` class with:

- Attributes: `name`, `email`, `tier`, `points` (starts at 0).
- Method `earn_points(self, amount_spent)` — adds 1 point per dollar spent.
- Method `redeem_points(self, points_to_redeem)` — subtracts points and returns the dollar value (1 point = $0.01). Should raise a `ValueError` if the customer doesn't have enough points.
- Method `check_tier_up(self)` — upgrades the customer's tier based on lifetime points earned. You decide the thresholds.

After you've built it, run the test cell to confirm it works.

In [ ]:
class LoyaltyCustomer:
    # TODO: your implementation
    pass


# Test (write your own — at minimum, exercise earn_points, redeem_points, and check_tier_up)


## At-home — marked pass/fail on engagement (L5): the purchase-history design choice

You've been asked to track which products each customer has purchased over time. **Where should this information live?**

There are two reasonable designs:

- **Design A:** Put a `purchases` list on `Customer`. The Customer object knows its own history.
- **Design B:** Put a `customer` reference on each `Order`. The Order objects collectively form the history; the Customer doesn't carry it.

This task has **no correct answer**. We're grading whether you engage with the design question seriously — not which choice you make.

Your task:

1. Implement **both** designs in the two code cells below. Keep them small — just enough to show the difference.
2. In the markdown cell after, argue for one design and against the other in 4–5 sentences. Be specific about the situations where one is better than the other.

In [ ]:
# Design A: purchases live on Customer
class CustomerA:
    # TODO: your implementation
    pass

# Sketch enough to demonstrate the design

In [ ]:
# Design B: purchases inferred from Order objects
class CustomerB:
    # TODO: your implementation
    pass

class OrderB:
    # TODO: your implementation — has a customer reference
    pass

# Sketch enough to demonstrate the design

**Your argument (4–5 sentences):**

*(replace this text)*

## Reflection — Part 2 (required)

Answer all three. This is short — a sentence or two each is fine.

1. **What was the hardest part of Part 2?** Not the typing — the *concept*.
2. **Pick one line of code you wrote and explain what's happening to a hypothetical teammate who knows S4 but not S5.** Three sentences max.
3. **If you used Claude or another AI tool:** paste your prompt(s). Then one sentence on what it got right, and one sentence on anything you changed, rejected, or didn't understand and had to look up.

**Your reflection:**

*(replace this text)*

---

# Part 3: `isinstance` and type checking

Sometimes a function needs to handle different kinds of inputs differently. `isinstance()` lets you check what type of object you're holding.

## Instructor demo — read

Here are three classes and a function that uses `isinstance` to distinguish them:

In [ ]:
class Shape:
    pass

class Circle(Shape):
    pass

class Square(Shape):
    pass


def identify_shape(s):
    if isinstance(s, Circle):
        return "This is a circle."
    elif isinstance(s, Square):
        return "This is a square."
    else:
        return "This is a shape of unknown type."


circle = Circle()
square = Square()

print(identify_shape(circle))
print(identify_shape(square))

**Notice:** `Circle(Shape)` and `Square(Shape)` mean Circle and Square are *kinds of* Shape. You'll meet this idea properly next session — for now, just know that this is the syntax.

## Predict-then-run (L2)

Before running the next cell, predict what each of the three `isinstance` calls returns. Write your predictions in the markdown cell first.

**Your predictions:**

- `isinstance(circle, Circle)` → ?
- `isinstance(circle, Shape)` → ?
- `isinstance(circle, object)` → ?

*(replace this text with your predictions)*

In [ ]:
print("isinstance(circle, Circle):", isinstance(circle, Circle))
print("isinstance(circle, Shape): ", isinstance(circle, Shape))
print("isinstance(circle, object):", isinstance(circle, object))

The third one is surprising the first time you see it. **Every** Python value is an `object` — `object` is the most general type there is. The point of this surprise: `isinstance` tests against a *family* of types, not just the most specific one. A Circle is a Circle, a Shape, AND an object — all true at once.

## Your turn (L3): add a Triangle

Extend the code so:

1. `Triangle` exists as a class derived from `Shape`.
2. `identify_shape` returns `"This is a triangle."` for a Triangle instance.

In [ ]:
# TODO: define Triangle


def identify_shape(s):
    # TODO: extend this function
    if isinstance(s, Circle):
        return "This is a circle."
    elif isinstance(s, Square):
        return "This is a square."
    else:
        return "This is a shape of unknown type."


# Test
print(identify_shape(Triangle()))   # should print "This is a triangle."
print(identify_shape(Circle()))     # should still print 'This is a circle.'


## Find the bug (L3): `type()` vs `isinstance()`

Below is an alternative version of `identify_shape` that uses `type(s) == Circle` instead of `isinstance(s, Circle)`. It looks like it should do the same thing. **It does not.** Run the cells and find the difference.

In [ ]:
def identify_shape_buggy(s):
    if type(s) == Circle:
        return "This is a circle."
    elif type(s) == Square:
        return "This is a square."
    else:
        return "This is a shape of unknown type."


# A more specific kind of Circle
class FancyCircle(Circle):
    pass


fancy = FancyCircle()

print("isinstance version:", identify_shape(fancy))         # works
print("type() version:    ", identify_shape_buggy(fancy))   # fails

**Your task:** explain in one or two sentences why the `type() ==` version fails on a `FancyCircle`, when the `isinstance` version handles it correctly.

**Your explanation:**

*(replace this text)*

## At-home — Claude allowed (L4): `validate_cart_item`

Build a function `validate_cart_item(item)` that accepts cart items and returns one of:

- the item unchanged, if it's a valid `Product` instance (define a small `Product` class with `id`, `name`, `price`)
- a clear error message string, if the item is a raw `str` ("we don't accept manually-typed item names")
- a different clear error message string, if the item is a raw `dict` ("we used to accept dicts in S4 but we don't anymore")
- a generic error message for anything else

The function should use `isinstance` to make the type checks.

In [ ]:
# TODO: your implementation


## At-home — marked pass/fail on engagement (L5): is your `isinstance` use legitimate?

You now have two functions that branch on `isinstance`:

- `identify_shape(s)` — branches on Circle / Square / Triangle.
- `validate_cart_item(item)` — branches on Product / str / dict / other.

For each function, decide: **is the `isinstance` branching a legitimate design, or is it a sign you wanted inheritance instead?** Argue your position in 3–4 sentences per function. Be specific.

(Hint: next session you'll meet inheritance properly. One of these two functions could be cleaned up with inheritance; the other genuinely needs `isinstance`. Make a call now and we'll revisit your reasoning next session.)

**Your argument:**

*(replace this text)*

---

# Part 4: Composition — building bigger things from smaller things

You now have a `Cart` class and a `Customer` class. **Where does an order live?** It's not a customer — a customer can place many orders. It's not a cart — a cart is just the items, not the buyer. An order is something bigger: it connects a customer to a cart.

This is **composition**: a class whose attributes are themselves objects of other classes.

## Instructor demo — read

In [ ]:
class Cart:
    def __init__(self, items):
        self.items = items

    def compute_subtotal(self, catalogue):
        total = 0
        for item in self.items:
            price = catalogue[item["product_id"]]["price"]
            total += price * item["qty"]
        return total


class Customer:
    def __init__(self, name, email, tier):
        self.name = name
        self.email = email
        self.tier = tier

    def apply_discount(self, total):
        if self.tier == "gold":   return total * 0.85
        if self.tier == "silver": return total * 0.90
        return total


class Order:
    def __init__(self, customer, cart):
        self.customer = customer    # composition: Order HAS A Customer
        self.cart = cart            # composition: Order HAS A Cart

    def compute_subtotal(self, catalogue):
        return self.cart.compute_subtotal(catalogue)   # delegate to the cart

    def compute_total(self, catalogue):
        subtotal = self.compute_subtotal(catalogue)
        return self.customer.apply_discount(subtotal)  # delegate to the customer


# Use it
alice = Customer("Alice Chen", "alice@example.com", "gold")
alice_cart = Cart(items=[
    {"product_id": "P001", "qty": 2},
    {"product_id": "P002", "qty": 1},
])
order = Order(alice, alice_cart)
print("Total:", order.compute_total(catalogue))

## Predict-then-run (L2)

Predict the output of each line below **before** running the next cell:

**Your predictions:**

- `order.cart` → ?
- `order.cart.items` → ?
- `order.customer.name` → ?

*(replace this text with your predictions — for object references, just describe what kind of thing it is)*

In [ ]:
print("order.cart:         ", order.cart)
print("order.cart.items:   ", order.cart.items)
print("order.customer.name:", order.customer.name)

## Your turn (L3): Engine and Car

Build two classes that demonstrate composition:

- `Engine` has a method `start(self)` that returns the string `"Engine started."`.
- `Car` **has-an** Engine. Car has a method `start_car(self)` that calls the engine's `start()` method, then returns the string `"Engine started. Car started."`.

In [ ]:
class Engine:
    # TODO
    pass


class Car:
    # TODO — note: the Car needs an engine attribute
    pass


# Test
my_car = Car()
print(my_car.start_car())   # should print 'Engine started. Car started.'


## Extend: add a Battery

Now your customer wants an electric car. Add a `Battery` class with a `recharge(self)` method that returns `"Battery recharging."` (the engine should still start the same way — assume hybrid).

Modify `Car` so it has **both** an engine and a battery. Where does the `recharge()` method belong — on Car, or on Battery? Make a choice and justify it.

In [ ]:
class Battery:
    # TODO
    pass


class CarV2:
    def __init__(self):
        # TODO: has an engine AND a battery
        pass

    # TODO: where does recharge() go?


# Test however makes sense given your design


**Justification for where `recharge()` lives:**

*(replace this text — 2 sentences are fine)*

## Refactor (L3): eager total vs lazy total

Below is a version of `Order` that stores the total as an attribute. It looks fine — until the cart's contents change. Run the cell and look at what goes wrong.

In [ ]:
class OrderEager:
    def __init__(self, customer, cart, catalogue):
        self.customer = customer
        self.cart = cart
        self.total = cart.compute_subtotal(catalogue)   # computed once, at construction


alice_cart = Cart(items=[{"product_id": "P001", "qty": 2}])
order = OrderEager(alice, alice_cart, catalogue)
print("Initial total:", order.total)

# Customer adds another item
alice_cart.items.append({"product_id": "P003", "qty": 1})
print("After adding stapler — cart items:", alice_cart.items)
print("After adding stapler — order.total:", order.total)   # STALE

**Your task:** refactor the class so the total is computed **on demand** (every time you ask for it), not stored as an attribute. Hint: turn `total` into a method, not an attribute.

(This is a classic OOP design lesson: derived data — values computed from other values — usually shouldn't be stored as attributes, because then you have to remember to update them whenever the source data changes. Methods that compute on demand stay correct automatically.)

In [ ]:
class OrderLazy:
    # TODO
    pass


# Test
alice_cart = Cart(items=[{"product_id": "P001", "qty": 2}])
order = OrderLazy(alice, alice_cart, catalogue)
print("Initial total:", order.compute_total(catalogue))   # or however you've named it

alice_cart.items.append({"product_id": "P003", "qty": 1})
print("After adding stapler — total:", order.compute_total(catalogue))   # should be up to date

## Comprehension check

Look at this expression: `order.cart.items`.

- What is `order.cart`? (What kind of object?)
- What is `order.cart.items`? (What kind of value?)
- Which of `order.cart` and `order.cart.items` represents the *composition* relationship?

**Your answer:**

*(replace this text)*

## At-home — Claude allowed (L4): a Library

Build a small library system using composition.

- A `Book` class with `title`, `author`, and `is_borrowed` (bool, starts False).
- A `Member` class with `name` and `borrowed_books` (a list, starts empty).
- A `Library` class that has-a list of Books and has-a list of Members. It needs methods:
  - `borrow(self, member, book)` — marks the book as borrowed, adds it to the member's borrowed list. Raises `ValueError` if the book is already borrowed.
  - `return_book(self, member, book)` — the reverse.

In [ ]:
# TODO: your Library system


## At-home — marked pass/fail on engagement (L5): could the Library have been built differently?

There's a phrase you'll hear often in OOP: **"favour composition over inheritance."** You've now built the Library with composition (Library has Books, Library has Members; Member has borrowed_books).

Could it have been built with inheritance instead? Sketch an alternative design in a few lines below (no need to fully implement it). For example: would `BorrowingMember(Member)` be a better way to express borrowing behaviour? Would `BorrowedBook(Book)` be a class?

Then argue: **which design do you prefer, and why?** 4–5 sentences. There is no correct answer.

In [ ]:
# Sketch of an inheritance-based alternative (a few lines, no need to fully implement)



**Your argument:**

*(replace this text)*

---

# Part 5: Mini-project — Order Processor (OO version)

Now you'll bring everything together. Your task is to **refactor your S4 mini-project into classes.**

You can do the in-class portion of this with a partner. The L5 reflection at the end is solo.

## The brief

In S4, you had:

- A `customer` dict and an `apply_discount` function.
- A `cart` dict and a `compute_subtotal` function.
- A `catalogue` dict.
- A `process_order(customer, cart, catalogue)` function that called the others and returned an order summary dict.

In S5, you'll have:

- A `Customer` class (already built in Part 2).
- A `Cart` class (already built in Part 1).
- An `Order` class with a `process(self, catalogue)` method that returns the order summary dict.

The **logic** stays identical. Only the **structure** changes.

In [ ]:
class Cart:
    def __init__(self, items):
        self.items = items

    def compute_subtotal(self, catalogue):
        total = 0
        for item in self.items:
            price = catalogue[item["product_id"]]["price"]
            total += price * item["qty"]
        return total


class Customer:
    def __init__(self, name, email, tier):
        self.name = name
        self.email = email
        self.tier = tier

    def apply_discount(self, total):
        if self.tier == "gold":   return total * 0.85
        if self.tier == "silver": return total * 0.90
        return total


class Order:
    def __init__(self, customer, cart):
        self.customer = customer
        self.cart = cart

    def process(self, catalogue):
        # TODO: compute subtotal (via the cart),
        #       apply the customer's discount,
        #       return a summary dict like:
        # {
        #     "customer_name": ...,
        #     "items": ...,
        #     "subtotal": ...,
        #     "total": ...,
        # }
        pass


# Test
catalogue = {
    "P001": {"name": "Notebook", "price": 12.50},
    "P002": {"name": "Pen pack", "price": 4.20},
    "P003": {"name": "Stapler", "price": 8.90},
}

alice = Customer("Alice Chen", "alice@example.com", "gold")
alice_cart = Cart(items=[
    {"product_id": "P001", "qty": 2},
    {"product_id": "P002", "qty": 1},
])
order = Order(alice, alice_cart)

summary = order.process(catalogue)
print(summary)

## At-home — marked pass/fail on engagement (L5): line-by-line comparison with S4

Open your Session 4 notebook in a second tab. Compare your S4 `process_order` function and your S5 `Order.process` method side by side.

Identify:

- **Three things that became easier or cleaner** in the S5 version. Be specific — paste the relevant snippets from each version in markdown and point at the difference.
- **One thing that became harder, less convenient, or more verbose** in the S5 version. (There usually is at least one. Honest answers are valued.)

This is graded on engagement: a reflective comparison with concrete snippets is what we want. Hand-waving is not.

**Your comparison:**

*(replace this text — use markdown code blocks for snippets, e.g.:*

```python
# S4:
def process_order(customer, cart, catalogue): ...
```

```python
# S5:
class Order:
    def process(self, catalogue): ...
```

)*

## Final reflection (required)

Three short answers:

1. **What was the hardest concept in Session 5?** Not the syntax — the idea. Was it `self`? Composition vs. just storing references? When to use `isinstance`? Something else?

2. **Pick one method you wrote in this notebook and explain it to a teammate who knows S4 but hasn't seen S5 yet.** Use the actual code — paste the snippet — and walk through what each line does. Three sentences max.

3. **If you used Claude or another AI tool in any L4 or L5 task:** paste the prompt you used (the most useful one if you used many). Then: one sentence on what it got right, and one sentence on anything you changed, rejected, or had to look up because the AI's answer didn't match what we'd done in class.

**Your final reflection:**

*(replace this text)*

---

## What's coming next: Session 6

Today you learned how to **build** classes (S5 foundations). Next session you'll learn how to **extend** them:

- **Inheritance.** When you have several similar classes (a Book, an Electronic, an Apparel item) that share most of their behaviour, Python lets one class *inherit* from another so you don't repeat yourself.
- **Magic methods.** Special methods like `__str__` and `__eq__` that let your objects work seamlessly with `print()`, `==`, `len()`, and Python's other built-in operations.
- **`@dataclass`.** A Python decorator that writes most of `__init__` for you. Massive boilerplate reduction.

A short comparison preview:

| What you learned today (S5) | What's next (S6) |
|---|---|
| Composition — an Order **has a** Customer | Inheritance — a Book **is a** Product |
| `__init__` and writing it yourself | `@dataclass` — Python writes `__init__` for you |
| Using `isinstance` to branch by type | Letting inheritance handle the branching for you |

**Before Session 6:** complete the at-home L4 and L5 cells in Parts 2–5. Read the *Job Ready Python* inheritance chapter.